# Form Saathi: fine-tune the jargon simplifier (QLoRA)

Teaches a small multilingual model to turn official form language into plain **English, Tamil, Hindi and Odia**.

* Base model: `Qwen/Qwen2.5-1.5B-Instruct` (multilingual, fits a free Colab T4 in 4-bit)
* Method: QLoRA (4-bit NF4 base + LoRA adapters on all attention and MLP projections)
* Loss only on the answer tokens (prompt tokens are masked with -100)
* Evaluation: the base model and the fine-tuned model on the same held-out sentences (chrF, jargon rate, correct script, length)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. It takes about 30-40 minutes.
At the end two files download: the adapter (`formsaathi-simplifier-lora.zip`) and `eval_results.json`.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

### Get the dataset and the eval code from the repo
`data/train.jsonl` and `data/val.jsonl` are built by `python build_dataset.py --augment 425` and committed to the repo.
The validation set is split by source sentence, so the model never sees a validation sentence during training.

In [ ]:
import os, sys
BRANCH = "claude/project-thread-w3dub6"  # falls back to main once this branch is merged
if not os.path.exists("form-saathi"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/surbhi1702-dev/form-saathi.git || git clone -q --depth 1 https://github.com/surbhi1702-dev/form-saathi.git
sys.path.insert(0, "form-saathi/finetune")
DATA = "form-saathi/finetune/data"

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 384

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows, val_rows = load(f"{DATA}/train.jsonl"), load(f"{DATA}/val.jsonl")
print(len(train_rows), "train /", len(val_rows), "val")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

### Tokenise with the chat template and mask the prompt
Only the assistant's plain-language answer contributes to the loss, so the model learns to *write simply* rather than to repeat the jargon.

In [ ]:
def encode(row):
    msgs = row["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True, tokenize=False)
    full = tok.apply_chat_template(msgs, tokenize=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = ([-100] * len(p_ids) + f_ids[len(p_ids):])[:len(f_ids)]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels}

train_ds = Dataset.from_list(train_rows).map(encode, remove_columns=list(train_rows[0].keys()))
val_ds = Dataset.from_list(val_rows).map(encode, remove_columns=list(val_rows[0].keys()))

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (n - len(seq))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch])}

### Generation helper (batched, greedy)

In [ ]:
SYSTEM = train_rows[0]["messages"][0]["content"]

def generate(batch, max_new_tokens=120, bs=16):
    """batch: list of (text, lang) -> list of plain-language outputs."""
    tok.padding_side = "left"
    model.eval()
    model.config.use_cache = True
    outs = []
    for i in range(0, len(batch), bs):
        prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                            {"role": "user", "content": f"[{lang}] {text}"}],
                                           add_generation_prompt=True, tokenize=False) for text, lang in batch[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                                 pad_token_id=tok.pad_token_id)
        outs += [o.strip() for o in tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    tok.padding_side = "right"
    return outs

def simplify(text, lang):
    return generate([(text, lang)])[0]

### Baseline: the base model before training
LoRA adapters start at zero, so the untrained model gives the same answers as the base model. We score it now for a fair before/after.

In [ ]:
from eval_simplifier import evaluate, report

results = {"base": evaluate(generate, val_rows)}
report(results)
for lang in ["en", "ta", "hi", "or"]:
    print(lang, "->", simplify("The applicant shall furnish a self-attested copy of the encumbrance certificate.", lang))

### Train

In [ ]:
import math
model.config.use_cache = False
EPOCHS, BATCH, ACCUM = 3, 8, 2
total_steps = math.ceil(len(train_ds) / (BATCH * ACCUM)) * EPOCHS
args = TrainingArguments(
    output_dir="out", per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM, num_train_epochs=EPOCHS, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=max(1, total_steps // 20), fp16=True, logging_steps=10, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
trainer.train()

### Evaluate the fine-tuned model on the same held-out sentences

In [ ]:
results["finetuned"] = evaluate(generate, val_rows)
results["train_log"] = {"train_rows": len(train_rows), "val_rows": len(val_rows),
                        "eval_loss": [round(h["eval_loss"], 4) for h in trainer.state.log_history if "eval_loss" in h]}
report({k: v for k, v in results.items() if k != "train_log"})
print(results["train_log"])

hyps = generate([(r["messages"][1]["content"].split("] ", 1)[1], r["lang"]) for r in val_rows[:8]])
for r, h in zip(val_rows[:8], hyps):
    print(f"\n[{r['lang']}] {r['messages'][1]['content'].split('] ', 1)[1]}\n  model: {h}\n  ref:   {r['messages'][2]['content']}")

### Save the adapter (about 40 MB) and the results, and download them
Copy the adapter folder next to `serve_simplifier.py` and run `python serve_simplifier.py`.

In [ ]:
model.save_pretrained("formsaathi-simplifier-lora")
tok.save_pretrained("formsaathi-simplifier-lora")
json.dump(results, open("eval_results.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
!zip -qr formsaathi-simplifier-lora.zip formsaathi-simplifier-lora
from google.colab import files
files.download("eval_results.json")
files.download("formsaathi-simplifier-lora.zip")

### Optional: publish the adapter on Hugging Face
Gives the model a public page. Make a **write** token at huggingface.co → Settings → Access Tokens, paste it when asked,
and set `HF_REPO` to `your-username/formsaathi-simplifier-lora`. The app can then load it with
`python serve_simplifier.py --adapter your-username/formsaathi-simplifier-lora`.

In [ ]:
HF_REPO = ""  # e.g. "your-username/formsaathi-simplifier-lora"
if HF_REPO:
    from huggingface_hub import login
    login()
    model.push_to_hub(HF_REPO, private=False)
    tok.push_to_hub(HF_REPO)
    print("https://huggingface.co/" + HF_REPO)